# Evaluate UniDepthLSS

Evaluate vehicle IoU at `448 × 798`, both without a visibility filter and with the NuScenes visibility filter (`visibility >= 2`). Both metrics use the same predictions in one pass.

In [ ]:
import sys
from pathlib import Path

import torch
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

# Data and checkpoint paths
DATA_ROOT = Path("/data/adeel/data/nuscenes")
CHECKPOINT = Path("/home/adeel/UniDepth_BEV/27_May-VGGTBeV/checkpoints/best_model.pt")
UNIDEPTH_ROOT = Path("/home/adeel/UniDepth")
sys.path.insert(0, str(UNIDEPTH_ROOT))

from dataset_nuscenes import NuScenesBEVDataset
from model import UniDepthLSS
from train_utils import BinaryIoU

IMAGE_SIZE = (448, 798)
BEV_SIZE = 128
BEV_RESOLUTION = 0.5
FEATURE_CHANNELS = 128
MIN_VISIBILITY = 2
THRESHOLD = 0.5
BATCH_SIZE = 1
NUM_WORKERS = 4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

for path in (DATA_ROOT, CHECKPOINT, UNIDEPTH_ROOT):
    if not path.exists():
        raise FileNotFoundError(path)

In [ ]:
dataset = NuScenesBEVDataset(
    dataroot=str(DATA_ROOT), version="v1.0-trainval", split="val",
    img_size=IMAGE_SIZE, bev_size=BEV_SIZE, bev_res=BEV_RESOLUTION,
    augment=False, return_visibility=True,
)
loader = DataLoader(
    dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
    pin_memory=DEVICE.type == "cuda",
)

model = UniDepthLSS(
    img_height=IMAGE_SIZE[0], img_width=IMAGE_SIZE[1],
    num_classes=1, feature_channels=FEATURE_CHANNELS,
).to(DEVICE)
checkpoint = torch.load(CHECKPOINT, map_location=DEVICE, weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

In [ ]:
without_visibility_filter = BinaryIoU(threshold=THRESHOLD)
with_visibility_filter = BinaryIoU(threshold=THRESHOLD)

with torch.inference_mode():
    for images, intrinsics, extrinsics, target, visibility in tqdm(loader):
        images = images.to(DEVICE, non_blocking=True)
        intrinsics = intrinsics.to(DEVICE, non_blocking=True)
        extrinsics = extrinsics.to(DEVICE, non_blocking=True)
        target = target.to(DEVICE, non_blocking=True)
        visibility = visibility.to(DEVICE, non_blocking=True)
        with torch.autocast(
            device_type=DEVICE.type, dtype=torch.float16, enabled=DEVICE.type == "cuda"
        ):
            logits = model(images, intrinsics, extrinsics)
        without_visibility_filter.update(logits, target)
        with_visibility_filter.update(
            logits, target, valid_mask=visibility >= MIN_VISIBILITY
        )

print(f"Without Visibility Filter — Vehicle IoU: {without_visibility_filter.compute():.4f}")
print(f"With Visibility Filter    — Vehicle IoU: {with_visibility_filter.compute():.4f}")